In [2]:
# Actividad 13 - PASO 1: Cargar datasets crudos

import pandas as pd
import numpy as np
import re

df_candidate = pd.read_csv("candidate_profile.csv", encoding='utf-8-sig')
df_dictionary = pd.read_csv("data_dictionary.csv", encoding='utf-8-sig')
df_demographics = pd.read_csv("demographics.csv", encoding='utf-8-sig')
df_plantilla = pd.read_csv("plantilla_global.csv", encoding='utf-8-sig')
df_succession = pd.read_csv("succession.csv", encoding='utf-8-sig')

In [3]:
# PASO 2: Actividad 4 — estandarizar nombres a snake_case, fijar índice, crear nomination_id


def to_snake_case(col):
    col = col.strip()
    col = col.replace('.', ' ')
    col = re.sub(r'[^\w\s]', '', col)
    col = re.sub(r'\s+', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.lower().strip('_')

for df in [df_candidate, df_demographics, df_plantilla, df_succession]:
    df.columns = [to_snake_case(c) for c in df.columns]

# Índice: Employee ID / NIP como identificador único por tabla (Actividad 4, P2-P3)
df_plantilla = df_plantilla.set_index('nip', drop=False)
df_demographics = df_demographics.set_index('employee_id', drop=False)
df_candidate = df_candidate.set_index('employee_id', drop=False)

# Succession no tiene fila única -> se crea nomination_id (antes "Indice"), 1 por evento de nominación
df_succession.insert(0, 'nomination_id', range(1, len(df_succession) + 1))

print("Columnas estandarizadas. Ejemplo Plantilla Global:", df_plantilla.columns.tolist()[:6])
print("nomination_id creado, rango:", df_succession['nomination_id'].min(), "-", df_succession['nomination_id'].max())

Columnas estandarizadas. Ejemplo Plantilla Global: ['nip', 'nombre', 'genero', 'f_nacimiento', 'f_i_alfa', 'f_i_sigma']
nomination_id creado, rango: 1 - 38200


In [4]:
# PASO 3: Actividad 5 — tipos de fecha, identificadores como texto, variables derivadas

HOY = pd.Timestamp.now().normalize()

# --- Fechas a datetime ---
for col in ['date_of_birth', 'date_of_entry_to_alfa', 'date_of_entry_to_sigma']:
    df_demographics[col] = pd.to_datetime(df_demographics[col], format='%Y-%m-%d', errors='coerce')

df_candidate['date_of_birth'] = pd.to_datetime(df_candidate['date_of_birth'], format='%Y-%m-%d', errors='coerce')
for col in ['prevjob_start_1', 'prevjob_end_1', 'prevjob_start_2', 'prevjob_end_2']:
    df_candidate[col] = pd.to_datetime(df_candidate[col], format='%Y-%m-%d', errors='coerce')

for col in ['f_nacimiento', 'f_i_alfa', 'f_i_sigma']:
    df_plantilla[col] = pd.to_datetime(df_plantilla[col], format='%Y-%m-%d', errors='coerce')

for col in ['incumbent_date_of_birth', 'nominee_date_of_birth', 'nomination_date']:
    df_succession[col] = pd.to_datetime(df_succession[col], format='%Y-%m-%d', errors='coerce')

# --- Identificadores que no son cantidades ---
df_candidate['optional_phone'] = df_candidate['optional_phone'].astype(str)
df_candidate['postal_code'] = df_candidate['postal_code'].astype(str)

# --- Variables derivadas (Demographics) ---
df_demographics['age_calculated'] = ((HOY - df_demographics['date_of_birth']).dt.days / 365.25).round(1)
df_demographics['tenure_sigma_calculated'] = ((HOY - df_demographics['date_of_entry_to_sigma']).dt.days / 365.25).round(1)
df_demographics['entry_sigma_year'] = df_demographics['date_of_entry_to_sigma'].dt.year
df_demographics['entry_sigma_month'] = df_demographics['date_of_entry_to_sigma'].dt.month

# --- Variables derivadas (Succession) ---
df_succession['nomination_year'] = df_succession['nomination_date'].dt.year
df_succession['nomination_month'] = df_succession['nomination_date'].dt.month
df_succession['years_to_retirement'] = (65 - (HOY - df_succession['incumbent_date_of_birth']).dt.days / 365.25).round(1)

# SUPUESTO (confírmalo): cortes de urgencia en años para retiro del titular
def clasificar_urgencia(years):
    if pd.isna(years): return np.nan
    if years <= 2: return 'Crítica'
    elif years <= 5: return 'Alta'
    elif years <= 10: return 'Media'
    else: return 'Baja'

df_succession['succession_urgency'] = df_succession['years_to_retirement'].apply(clasificar_urgencia)

print("Variables derivadas de Actividad 5 creadas.")
print(df_succession[['nomination_id', 'years_to_retirement', 'succession_urgency']].head())

Variables derivadas de Actividad 5 creadas.
   nomination_id  years_to_retirement succession_urgency
0              1                 22.3               Baja
1              2                 30.6               Baja
2              3                 29.0               Baja
3              4                 29.5               Baja
4              5                 36.3               Baja


In [5]:

# PASO 4: Actividad 9 — idioma como categórica ordinal


orden_niveles = ["Intermedio", "Avanzado", "Nativo"]
df_candidate['language_1_level'] = pd.Categorical(df_candidate['language_1_level'], categories=orden_niveles, ordered=True)
df_candidate['language_2_level'] = pd.Categorical(df_candidate['language_2_level'], categories=orden_niveles, ordered=True)

print("Language levels convertidos a categórica ordinal.")

Language levels convertidos a categórica ordinal.


In [6]:

# PASO 5: Actividad 10 — verificación de duplicados y outliers (decisión ya tomada: conservar)


for nombre, df in [('Candidate', df_candidate), ('Demographics', df_demographics),
                    ('Plantilla', df_plantilla), ('Succession', df_succession)]:
    print(f"{nombre}: {df.duplicated().sum()} duplicados")

print("\nOutliers de las 6 variables (Bonos, Puntajes, Antigüedad): conservados sin eliminar,")
print("decisión tomada y justificada en Actividad 10 — no se modifica ninguna fila aquí.")

Candidate: 0 duplicados
Demographics: 0 duplicados
Plantilla: 0 duplicados
Succession: 0 duplicados

Outliers de las 6 variables (Bonos, Puntajes, Antigüedad): conservados sin eliminar,
decisión tomada y justificada en Actividad 10 — no se modifica ninguna fila aquí.


In [7]:
# PASO 6: Actividad 11 — enriquecimiento (columnas derivadas, rangos, reagrupación)


# Rangos de sueldo (usando los cuartiles ya calculados en Actividad 6-7)
df_plantilla['salary_band'] = pd.cut(
    df_plantilla['sueldo_sap'], bins=[-np.inf, 6853.53, 77831.97, np.inf], labels=['Bajo', 'Medio', 'Alto']
)

# Rangos de score general
df_candidate['score_band'] = pd.cut(
    df_candidate['score_general'], bins=[-np.inf, 61.6, 67.3, np.inf], labels=['Bajo', 'Medio', 'Alto']
)

# Reagrupación de categorías: NIVEL (N1-N6) -> grupo jerárquico
# SUPUESTO (confírmalo): N1-N2 más alto IPE = Senior, N3-N4 = Medio, N5-N6 = Junior
mapa_nivel = {'n1': 'Senior', 'n2': 'Senior', 'n3': 'Medio', 'n4': 'Medio', 'n5': 'Junior', 'n6': 'Junior'}
df_plantilla['nivel_grupo'] = df_plantilla['nivel'].str.lower().map(mapa_nivel)

# Columna derivada: bono como % del sueldo (conecta con el hallazgo de Actividad 8)
df_plantilla['bono_pct_sueldo'] = (df_plantilla['bonos_y_com_ori'] / df_plantilla['sueldo_sap'] * 100).round(2)

print("Enriquecimiento aplicado: salary_band, score_band, nivel_grupo, bono_pct_sueldo")
print(df_plantilla[['nip', 'sueldo_sap', 'salary_band', 'nivel', 'nivel_grupo', 'bono_pct_sueldo']].head())

Enriquecimiento aplicado: salary_band, score_band, nivel_grupo, bono_pct_sueldo
                 nip  sueldo_sap salary_band nivel nivel_grupo  \
nip                                                              
NIP100001  NIP100001    17059.21       Medio    N3       Medio   
NIP100002  NIP100002     5885.81        Bajo    N3       Medio   
NIP100003  NIP100003     2977.94        Bajo    N6      Junior   
NIP100004  NIP100004    32281.28       Medio    N6      Junior   
NIP100005  NIP100005   117713.73        Alto    N1      Senior   

           bono_pct_sueldo  
nip                         
NIP100001             6.60  
NIP100002             5.49  
NIP100003             5.34  
NIP100004             6.57  
NIP100005             7.87  


In [10]:
# PASO 7: Construir la tabla maestra final (hecho + dimensiones)

df_master = df_succession.merge(
    df_plantilla[['nip', 'nivel', 'nivel_grupo', 'sueldo_sap', 'salary_band', 'bonos_y_com_ori', 'bono_pct_sueldo']].reset_index(drop=True),
    left_on='nominee_user_id', right_on='nip', how='left'
).merge(
    df_demographics[['employee_id', 'age', 'tenure_years_sigma']].reset_index(drop=True),
    left_on='nominee_user_id', right_on='employee_id', how='left', suffixes=('', '_demo')
).merge(
    df_candidate[['employee_id', 'score_general', 'score_band']].reset_index(drop=True),
    left_on='nominee_user_id', right_on='employee_id', how='left', suffixes=('', '_cand')
)

print(f"Tabla maestra final: {df_master.shape}")

Tabla maestra final: (38200, 37)


In [11]:
# ==========================================
# PASO 8: Exportar el dataset final
# ==========================================

df_candidate.to_csv("candidate_profile_final.csv", index=False)
df_demographics.to_csv("demographics_final.csv", index=False)
df_plantilla.to_csv("plantilla_global_final.csv", index=False)
df_succession.to_csv("succession_final.csv", index=False)
df_master.to_csv("tabla_maestra_final.csv", index=False)

print("Archivos finales exportados: candidate_profile_final.csv, demographics_final.csv,")
print("plantilla_global_final.csv, succession_final.csv, tabla_maestra_final.csv")

Archivos finales exportados: candidate_profile_final.csv, demographics_final.csv,
plantilla_global_final.csv, succession_final.csv, tabla_maestra_final.csv
